# Correct camera-vs-stage rotation

Each microscope's camera sits at a small fixed angle to the stage's X/Y axes.
`create_grid_positions` assumes none, so adjacent FOVs' overlaps don't line up
where the grid says, which can lose or duplicate cells/barcodes at FOV borders.

**Method**: sample a few anchor FOVs (default 10), register each against its
4-connected neighbours in their overlap border (phase correlation on a DAPI
frame of an already-imaged **cells round**), and fit ONE affine transform to
all correspondences (`MERci.acquisition.camera_rotation`, `affine6p`). The
rotation is a fixed optical property, so this transform corrects every FOV.

Single positions file only; the multi-boundary per-segment layout is not
handled.

**Outputs** (in `positions/`):
- `camera_rotation_correction_{POSITIONS_TAG}.npy`: the 3x3 affine matrix
- `camera_rotation_correction_{POSITIONS_TAG}.json`: fit metadata (rotation,
  scale, correspondence count, source round)
- `positions_{POSITIONS_TAG}_corrected.txt`: corrected FOV positions.
  `create_merlin_files.ipynb` uses it automatically when it exists.

Section 15 validates the fit on FOVs not used to compute it.

## 1 — Setup

In [ ]:
import os
import sys
import json
import yaml
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import KDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.io              import load_positions, save_positions_array, read_image_frames
from MERci.common.experiment_info import load_experiment_info, resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs    import (
    find_frame_table_for_hal_config, get_camera_pixel_size_um, get_camera_frame_size,
)
from MERci.acquisition.positions  import find_3x3_block
from MERci.acquisition.camera_rotation import (
    NeighborCorrespondence, sample_neighbor_correspondences, fit_camera_rotation,
    filter_correspondence_outliers, apply_microscope_orientation, crop_overlap,
    fit_global_positions,
)
from MERci.acquisition.alignment import remove_hot_pixels
from MERci.scheduler        import resolve_round_flip_y
from MERci.analysis.round   import load_raw_frames_for_round, create_mosaic_ffc
from MERci.progress_display import ProgressReporter
from MERci.plots.experiment_plots import get_merci_figures_dir

NOTEBOOK_NAME = "correct_camera_rotation"

print(f"SAMPLE_DIR : {SAMPLE_DIR}")


## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# From experiment_info.yaml (create_experiment_info). Pixel size and frame
# geometry are keyed on this exact string; a wrong one makes every registration fail.
info       = load_experiment_info(SAMPLE_DIR / "metadata" / "experiment_info.yaml")
MICROSCOPE = info.microscope

# Round to measure from, by imaging_type (usually "cells": every FOV, strong
# DAPI), or set ROUND_ID directly.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Frame-table CSV override, when metadata/ differs from the one used at
# acquisition. None = auto-resolve.
REFERENCE_FRAME_TABLE_PATH = None

# Registration colour (nm) and z (um). No default: pick a z with real tissue
# signal in REGISTRATION_COLOR_NM.
REGISTRATION_COLOR_NM = 405.0
REGISTRATION_Z_UM     = 12.0

# Anchor FOVs to register against their 4-connected neighbours (up to 4
# correspondences each), all pooled into one affine fit.
N_ANCHORS = 10

# Anchor sampling seed; None = new random sample each run.
SEED = 0

# Held-out validation (section 15): fresh anchors, disjoint from the fit's,
# with their own seed.
N_VALIDATION_ANCHORS = 10
VALIDATION_SEED      = 1

# Orientation audit (section 4): anchors to trial per orientation combination.
N_ORIENTATION_TRIAL_ANCHORS = 3

# Outlier rejection (sections 6 and 15): drop a correspondence whose shift
# exceeds median + MAD_THRESHOLD * 1.4826 * MAD.
MAD_THRESHOLD = 5.0

# Grid-neighbour match tolerance, as a fraction of step_size_um.
TOLERANCE_FRACTION = 0.25

# Subpixel precision (1/upsample_factor px).
UPSAMPLE_FACTOR = 10

# Zero the fitted translation: correspondences only measure local
# displacements, so a fitted translation is noise.
ZERO_TRANSLATION = True

# True = re-register even if cached correspondences exist (e.g. after
# changing N_ANCHORS/SEED/REGISTRATION_Z_UM).
FORCE_RECOMPUTE = False

# Half-width (um) of the zoomed corner-overlay crop (section 13). Tune if the
# crop looks empty.
CORNER_ZOOM_HALF_UM = 40.0

# Mosaic rendering (section 12). The overlap crop is computed there from the
# measured grid. thumbnail_size only applies to the 9x9 view.
MOSAIC_THUMBNAIL_SIZE = (200, 200)

# Half-width (um) of the zoom into each corner of the 3x3 mosaic (section 12).
MOSAIC_2X2_ZOOM_HALF_UM = 75.0

# Max exterior FOVs sampled for the flat-field field (section 11).
FFC_MAX_SAMPLES = 80

# Side of the FOV block in the 9x9 mosaic (section 12), centred on section
# 11's centre FOV. Reading every FOV over the network takes too long.
MOSAIC_9X9_N_FOVS = 9

# Plot font sizes.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Sample name   : {SAMPLE_NAME}")
print(f"Positions tag : {POSITIONS_TAG}")
print(f"Microscope    : {MICROSCOPE}")
print(f"N_ANCHORS     : {N_ANCHORS}")
print(f"N_VALIDATION_ANCHORS : {N_VALIDATION_ANCHORS}")

## 3 — Resolve the reference round + registration frame

In [ ]:
config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

print("Available rounds (data folders):")
for rid in meta.valid_round_ids():
    series_list = meta.series_for_round(rid)
    types = sorted({(s.imaging_type or "?") for s in series_list})
    print(f"  round {rid:>2}  imaging_type={types}")


def load_round_frame_table(round_id, config, meta, frame_table_path=None):
    """(frame_table, hal_config_path, SeriesInfo) for round_id's first series with a
    hal_config. Mirrors stage_drift_dapi.ipynb's own helper of the same name."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        if frame_table_path is not None:
            return pd.read_csv(frame_table_path, index_col=0), hal_path, s
        ft_path = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0), hal_path, s
    raise FileNotFoundError(f"No frame table found for round {round_id}")


def nearest_frame_for_color_z(frame_table, color_nm, z_um):
    """Frame index of the given colour's frame with z closest to z_um."""
    candidates = frame_table[frame_table["color"].round(0) == round(color_nm)]
    if candidates.empty:
        raise ValueError(f"No frames of colour {color_nm:.0f} nm in this frame table.")
    return int((candidates["z"] - z_um).abs().idxmin())


target_round_id = ROUND_ID if ROUND_ID is not None else meta.round_for_imaging_type(ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"some FOVs sampled below may be missing.")

if REGISTRATION_Z_UM is None:
    raise ValueError(
        "REGISTRATION_Z_UM is not set (section 2) -- pick a z (um) where "
        f"REGISTRATION_COLOR_NM ({REGISTRATION_COLOR_NM:.0f} nm) actually shows real "
        "signal (e.g. from measure_tissue_thickness_test.ipynb's z_first_um/z_last_um) "
        "before continuing."
    )

frame_table, reference_hal_config_path, reference_series = load_round_frame_table(
    target_round_id, config, meta, frame_table_path=REFERENCE_FRAME_TABLE_PATH,
)
reference_frame_idx = nearest_frame_for_color_z(frame_table, REGISTRATION_COLOR_NM, REGISTRATION_Z_UM)
reference_frame_z   = float(frame_table.loc[reference_frame_idx, "z"])

PIXEL_SIZE_UM     = get_camera_pixel_size_um(MICROSCOPE)
FRAME_WIDTH_PX, _ = get_camera_frame_size(MICROSCOPE)
FRAME_WIDTH_UM    = FRAME_WIDTH_PX * PIXEL_SIZE_UM
FLIP_Y            = resolve_round_flip_y(target_round_id, config, meta)

full_positions = load_positions(config.positions_txt)   # {fov_id: (x, y)} -- nominal grid
fov_order = sorted(full_positions)

# Step size and overlap measured from the real positions (median nearest-
# neighbour distance), not from ExperimentConfig's formula.
coords_arr       = np.array([full_positions[f] for f in fov_order])
nn_dist, _       = KDTree(coords_arr).query(coords_arr, k=2)
STEP_SIZE_UM     = float(np.median(nn_dist[:, 1]))
OVERLAP_FRACTION = max(0.0, 1.0 - STEP_SIZE_UM / FRAME_WIDTH_UM)

print(f"Target round      : {target_round_id}  (series pattern: {reference_series.name!r})")
print(f"Reference frame    : index {reference_frame_idx}, colour {REGISTRATION_COLOR_NM:.0f} nm, "
      f"z={reference_frame_z:.2f} um (requested z={REGISTRATION_Z_UM:.2f} um)")
print(f"Pixel size (um)    : {PIXEL_SIZE_UM}")
print(f"Frame width (um)   : {FRAME_WIDTH_UM:.3f}  ({FRAME_WIDTH_PX} px)")
print(f"Step size (um)     : {STEP_SIZE_UM:.3f}  (measured, median nearest-neighbour distance)")
print(f"Overlap fraction   : {OVERLAP_FRACTION:.3f}")
print(f"flip_y             : {FLIP_Y}")
print(f"FOVs in this round : {len(full_positions)}")


def load_dapi_frame(fov_id):
    path = reference_series.resolve_path(fov_id, config.image_suffix)
    return read_image_frames(path, [reference_frame_idx],
                              frame_width=config.frame_width, frame_height=config.frame_height)[0]


## 4 — Load this microscope's orientation from its MERlin microscope YAML

`crop_overlap` assumes rows follow stage +y and columns stage +x, with no flip.
The real `transpose`/`flip_horizontal`/`flip_vertical` values come from the
microscope's MERlin YAML (`data/configs/merlin/microscope/*.yaml`) and are
applied here.

`detect_image_orientation` tries all 8 combinations only as a cross-check of
the YAML.

In [ ]:
from MERci.acquisition.camera_rotation import detect_image_orientation
from MERci.acquisition.merlin_config import resolve_microscope_parameters_filename

microscope_params_path = (
    MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
    / resolve_microscope_parameters_filename(MICROSCOPE)
)
microscope_params = yaml.safe_load(microscope_params_path.read_text())
ORIENT_TRANSPOSE       = bool(microscope_params.get("transpose", False))
ORIENT_FLIP_HORIZONTAL = bool(microscope_params.get("flip_horizontal", False))
ORIENT_FLIP_VERTICAL   = bool(microscope_params.get("flip_vertical", False))

print(f"Microscope parameters : {microscope_params_path}")
print(f"  transpose       = {ORIENT_TRANSPOSE}")
print(f"  flip_horizontal = {ORIENT_FLIP_HORIZONTAL}")
print(f"  flip_vertical   = {ORIENT_FLIP_VERTICAL}")

best_combination, orientation_scan_df = detect_image_orientation(
    fov_ids=fov_order,
    positions=full_positions,
    load_frame=load_dapi_frame,
    step_size_um=STEP_SIZE_UM,
    pixel_size_um=PIXEL_SIZE_UM,
    overlap_fraction=OVERLAP_FRACTION,
    n_trial_anchors=N_ORIENTATION_TRIAL_ANCHORS,
    tolerance_fraction=TOLERANCE_FRACTION,
    upsample_factor=UPSAMPLE_FACTOR,
    seed=SEED,
)
print("\nAudit -- all 8 (transpose, flip_horizontal, flip_vertical) combinations:")
print(orientation_scan_df.to_string(index=False))

config_mask = (
    (orientation_scan_df["transpose"] == ORIENT_TRANSPOSE)
    & (orientation_scan_df["flip_horizontal"] == ORIENT_FLIP_HORIZONTAL)
    & (orientation_scan_df["flip_vertical"] == ORIENT_FLIP_VERTICAL)
)
config_rank         = int(orientation_scan_df.index[config_mask][0])
config_median_shift = float(orientation_scan_df.loc[config_mask, "median_shift_um"].iloc[0])
print(f"\nMicroscope-parameters combination ranks #{config_rank + 1}/8 by median shift "
      f"({config_median_shift:.2f} um).")

if best_combination != (ORIENT_TRANSPOSE, ORIENT_FLIP_HORIZONTAL, ORIENT_FLIP_VERTICAL):
    print(f"WARNING: the empirically best combination {best_combination} differs from "
          f"the microscope-parameters YAML's declared "
          f"({ORIENT_TRANSPOSE}, {ORIENT_FLIP_HORIZONTAL}, {ORIENT_FLIP_VERTICAL}). "
          "Using the YAML's value regardless below -- it's MERlin's own verified "
          "convention -- but investigate this disagreement before trusting the fit.")
else:
    print("Matches the empirically best combination -- consistent.")


## 5 — Sample neighbour correspondences (calculation, cached)

Registers `N_ANCHORS` random FOVs against their 4-connected neighbours on the
overlap border crop. Cached to
`analysis/cache/correct_camera_rotation/neighbor_correspondences.csv`; set
`FORCE_RECOMPUTE=True` to redo it.

In [ ]:
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
correspondences_csv = CACHE_DIR / "neighbor_correspondences.csv"

if correspondences_csv.exists() and not FORCE_RECOMPUTE:
    correspondences_df = pd.read_csv(correspondences_csv)
    print(f"Loaded cached correspondences: {correspondences_csv}  ({len(correspondences_df)} rows)")
else:
    n_anchors_actual = min(N_ANCHORS, len(fov_order))
    reporter = ProgressReporter(total=n_anchors_actual * 4, label="Registering neighbour pairs")

    def _progress(done, total):
        reporter.update(done - reporter.n_done)

    correspondences = sample_neighbor_correspondences(
        fov_ids=fov_order,
        positions=full_positions,
        load_frame=load_dapi_frame,
        step_size_um=STEP_SIZE_UM,
        pixel_size_um=PIXEL_SIZE_UM,
        overlap_fraction=OVERLAP_FRACTION,
        n_anchors=N_ANCHORS,
        tolerance_fraction=TOLERANCE_FRACTION,
        upsample_factor=UPSAMPLE_FACTOR,
        orient_transpose=ORIENT_TRANSPOSE,
        orient_flip_horizontal=ORIENT_FLIP_HORIZONTAL,
        orient_flip_vertical=ORIENT_FLIP_VERTICAL,
        seed=SEED,
        progress_callback=_progress,
    )
    reporter.done()

    correspondences_df = pd.DataFrame([{
        "anchor_fov": c.anchor_fov, "neighbor_fov": c.neighbor_fov, "direction": c.direction,
        "nominal_x": c.nominal_xy[0], "nominal_y": c.nominal_xy[1],
        "measured_x": c.measured_xy[0], "measured_y": c.measured_xy[1],
        "error": c.error,
    } for c in correspondences])
    correspondences_df.to_csv(correspondences_csv, index=False)
    print(f"Sampled {len(correspondences_df)} neighbour correspondence(s) from "
          f"{n_anchors_actual} anchor FOV(s). Saved: {correspondences_csv}")

print(correspondences_df.head())


## 6 — Fit the camera-rotation affine transform

In [ ]:
correspondences_all = [
    NeighborCorrespondence(
        anchor_fov=int(r.anchor_fov), neighbor_fov=int(r.neighbor_fov), direction=r.direction,
        nominal_xy=(r.nominal_x, r.nominal_y), measured_xy=(r.measured_x, r.measured_y), error=r.error,
    )
    for r in correspondences_df.itertuples()
]
correspondences, rejected = filter_correspondence_outliers(correspondences_all, mad_threshold=MAD_THRESHOLD)
print(f"Kept {len(correspondences)}/{len(correspondences_all)} correspondence(s) for the fit "
      f"({len(rejected)} rejected as outliers -- individual failed registrations, "
      f"the same 'bad link' problem BigStitcher's manual workflow curated by hand).")
for c in rejected:
    shift_um = float(np.hypot(c.measured_xy[0] - c.nominal_xy[0], c.measured_xy[1] - c.nominal_xy[1]))
    print(f"  rejected: anchor={c.anchor_fov} neighbor={c.neighbor_fov} "
          f"direction={c.direction!r} shift={shift_um:.2f} um")

correction = fit_camera_rotation(correspondences, zero_translation=ZERO_TRANSLATION)

M = correction.matrix
rotation_deg = float(np.degrees(np.arctan2(M[1, 0], M[0, 0])))
scale        = float(np.sqrt(M[0, 0] ** 2 + M[1, 0] ** 2))
print(f"Fitted from {correction.n_correspondences} correspondence(s):")
print(f"  rotation : {rotation_deg:.4f} deg")
print(f"  scale    : {scale:.6f}")
print(f"  matrix   :\n{M}")

positions_dir = SAMPLE_DIR / "positions"
matrix_path = positions_dir / f"camera_rotation_correction_{POSITIONS_TAG}.npy"
correction.save(matrix_path)

meta_path = positions_dir / f"camera_rotation_correction_{POSITIONS_TAG}.json"
with meta_path.open("w") as fh:
    json.dump({
        "n_correspondences": correction.n_correspondences,
        "zero_translation": correction.zero_translation,
        "rotation_deg": rotation_deg,
        "scale": scale,
        "source_round_id": target_round_id,
        "registration_color_nm": REGISTRATION_COLOR_NM,
        "registration_z_um": REGISTRATION_Z_UM,
    }, fh, indent=2)

print(f"Saved correction matrix  : {matrix_path}")
print(f"Saved correction metadata: {meta_path}")


## 7 — Apply the correction to the full positions file

In [ ]:
nominal_coords   = np.array([full_positions[f] for f in fov_order], dtype=float)
corrected_coords = correction.transform_points(nominal_coords)
corrected_positions = {f: tuple(corrected_coords[i]) for i, f in enumerate(fov_order)}

corrected_path = positions_dir / f"positions_{POSITIONS_TAG}_corrected.txt"
save_positions_array(corrected_coords, corrected_path)

print(f"Applied camera-rotation correction to all {len(full_positions)} FOV(s).")
print(f"Wrote corrected positions file: {corrected_path}")
print("Use this file as MERlin's positions input instead of the raw grid "
      "(create_merlin_files.ipynb auto-detects it).")

figures_dir = get_merci_figures_dir(SAMPLE_DIR, "misc", NOTEBOOK_NAME)
figures_dir.mkdir(parents=True, exist_ok=True)


## 8 -- Refine with a per-FOV global position optimization

The affine (section 6) can only correct a rotation/scale/shear shared by the
whole grid, not independent per-FOV stage jitter. `fit_global_positions` solves
each sampled FOV's position as a free variable against all its
correspondences, and overrides `corrected_positions` for those FOVs (the rest
keep the affine correction).

Limitation: with sparse anchor sampling each anchor and its neighbours form a
separate star, so each solved position just equals its one measurement
(`residual_rms_um` near 0). Averaging across measurements would need denser
sampling.

In [ ]:
global_position_result = fit_global_positions(correspondences, full_positions)

for fov_id, refined_xy in global_position_result.positions.items():
    corrected_positions[fov_id] = refined_xy
corrected_coords = np.array([corrected_positions[f] for f in fov_order], dtype=float)
save_positions_array(corrected_coords, corrected_path)

print(f"Refined {global_position_result.n_fovs_solved} FOV position(s) across "
      f"{global_position_result.n_components} independent correspondence-graph "
      f"component(s) ({global_position_result.n_correspondences} correspondence(s) total).")
print(f"Residual RMS: {global_position_result.residual_rms_um:.4f} um (near 0 is expected "
      "for this module's sparse, non-overlapping sampling -- see markdown above).")
print(f"Re-saved corrected positions file: {corrected_path}")


## 9 — Plot: original vs. corrected positions overlay

In [ ]:
shift_um = np.linalg.norm(corrected_coords - nominal_coords, axis=1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(nominal_coords[:, 0], nominal_coords[:, 1], "-", lw=0.3, color="0.75", zorder=1)
ax.plot(nominal_coords[:, 0], nominal_coords[:, 1], "o", ms=5, color="tab:blue",
        label="original", zorder=2)
ax.plot(corrected_coords[:, 0], corrected_coords[:, 1], "o", ms=5, mfc="none",
        mec="tab:red", mew=1.2, label="corrected", zorder=3)
ax.invert_yaxis()
ax.axis("equal")
ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Original vs. corrected FOV positions -- {SAMPLE_NAME}\n"
             f"rotation={rotation_deg:.3f} deg, scale={scale:.5f}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="upper right")

fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.positions_overlay.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")

**FOV shift distribution** -- per-FOV shift distance (original -> corrected) over all FOVs, which shows the correction even where it is too small to see in the overlay above.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.hist(shift_um, bins=30, color="tab:purple")
ax2.set_xlabel("shift (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax2.set_ylabel("FOV count", fontsize=PLOT_LABEL_FONTSIZE)
ax2.set_title(f"Per-FOV shift distribution (original -> corrected) -- {SAMPLE_NAME}\n"
              f"median {np.median(shift_um):.3f} um, max {shift_um.max():.3f} um",
              fontsize=PLOT_TITLE_FONTSIZE)
ax2.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig2.tight_layout()
fig2_path = figures_dir / f"{NOTEBOOK_NAME}.shift_histogram.png"
fig2.savefig(fig2_path, dpi=150)
plt.show()
print(f"Saved: {fig2_path}")

## 10 — Plot: original & corrected positions with FOV grids

In [ ]:
fov_size_um = FRAME_WIDTH_UM
half = fov_size_um / 2

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, coords, title in ((axes[0], nominal_coords, "original"), (axes[1], corrected_coords, "corrected")):
    for x, y in coords:
        ax.add_patch(mpatches.Rectangle((x - half, y - half), fov_size_um, fov_size_um,
                                         lw=0.3, edgecolor="tab:blue", facecolor="tab:blue", alpha=0.15))
    ax.plot(coords[:, 0], coords[:, 1], "o", ms=2, color="k")
    ax.invert_yaxis()
    ax.axis("equal")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"FOV footprints before/after camera-rotation correction -- {SAMPLE_NAME}",
             fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.fov_grids.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 11 — Prepare the 3x3 mosaic/overlay diagnostics: pick a centre FOV, compute a flat-field-correction field

Sections 12-13 share one complete 3x3 FOV neighbourhood and one flat-field
(FFC) field for `REGISTRATION_COLOR_NM`. Without FFC, vignetting darkens every
tile's border, which is where seams are judged. The field comes from up to
`FFC_MAX_SAMPLES` grid-exterior FOVs (`MERci.analysis.ffc`), cached.

In [ ]:
# Pick the centre among anchors/neighbours of kept training correspondences,
# which are known to have signal (FFC can't flatten a near-empty tile).
vetted_fov_ids = sorted({c.anchor_fov for c in correspondences} | {c.neighbor_fov for c in correspondences})
block_3x3 = find_3x3_block(vetted_fov_ids, full_positions, STEP_SIZE_UM, TOLERANCE_FRACTION)
if block_3x3 is None:
    print("No complete 3x3 block found among vetted (well-registered) FOVs -- falling back to the full grid.")
    block_3x3 = find_3x3_block(fov_order, full_positions, STEP_SIZE_UM, TOLERANCE_FRACTION)
if block_3x3 is None:
    raise RuntimeError("No complete 3x3 (8-connected) FOV neighbourhood found -- "
                        "try a larger TOLERANCE_FRACTION.")
anchor_fov = block_3x3["center"]
print(f"3x3 block centred on FOV {anchor_fov}: {block_3x3}")

# The fit applies ORIENT_* internally; this visualisation reads raw frames,
# so it applies the same orientation itself.
block_3x3_frames = {
    name: apply_microscope_orientation(load_dapi_frame(fov_id), transpose=ORIENT_TRANSPOSE,
                                        flip_horizontal=ORIENT_FLIP_HORIZONTAL, flip_vertical=ORIENT_FLIP_VERTICAL)
    for name, fov_id in block_3x3.items()
}

# Section 13's overlay uses the NE corner (centre/right/up/up_right).
block_names   = {"anchor": block_3x3["center"], "right": block_3x3["right"],
                  "up": block_3x3["up"], "diag": block_3x3["up_right"]}
block_fov_ids = list(block_names.values())
block_frames  = {"anchor": block_3x3_frames["center"], "right": block_3x3_frames["right"],
                  "up": block_3x3_frames["up"], "diag": block_3x3_frames["up_right"]}
print(f"Corner (NE) sub-block for section 12: {block_names}")


In [ ]:
from MERci.acquisition.positions import find_exterior_fovs
from MERci.analysis.ffc import compute_ffc_field_for_color, save_ffc_field, load_ffc_field

ffc_cache_path = CACHE_DIR / f"ffc_field_{REGISTRATION_COLOR_NM:.0f}nm.npz"

if ffc_cache_path.exists() and not FORCE_RECOMPUTE:
    ffc_field, ffc_meta = load_ffc_field(ffc_cache_path)
    print(f"Loaded cached FFC field: {ffc_cache_path}  (from {int(ffc_meta.get('n_samples'))} sample(s))")
else:
    exterior_fovs = sorted(find_exterior_fovs(full_positions, STEP_SIZE_UM, tolerance_fraction=TOLERANCE_FRACTION))
    if len(exterior_fovs) > FFC_MAX_SAMPLES:
        rng = np.random.default_rng(SEED)
        exterior_fovs = [int(f) for f in rng.choice(exterior_fovs, size=FFC_MAX_SAMPLES, replace=False)]
    ffc_samples = [
        (reference_series.resolve_path(fov_id, config.image_suffix), reference_frame_idx)
        for fov_id in exterior_fovs
    ]
    print(f"Computing FFC field from {len(ffc_samples)} exterior FOV(s) "
          f"(one frame read each -- may take a minute)...")
    ffc_field, ffc_meta = compute_ffc_field_for_color(
        ffc_samples, frame_width=config.frame_width, frame_height=config.frame_height,
    )
    save_ffc_field(ffc_cache_path, ffc_field, ffc_meta)
    print(f"Saved: {ffc_cache_path}")

# The field is cached in raw sensor space; orient it once to match the
# oriented frames.
ffc_field = apply_microscope_orientation(ffc_field, transpose=ORIENT_TRANSPOSE,
                                          flip_horizontal=ORIENT_FLIP_HORIZONTAL, flip_vertical=ORIENT_FLIP_VERTICAL)
print(f"FFC field: shape={ffc_field.shape}, min={ffc_field.min():.3f}, max={ffc_field.max():.3f}")


## 12 — Plot: original vs. corrected DAPI mosaic comparison

Original vs. corrected, flat-field-corrected and cropped to the measured
overlap border (`MOSAIC_CROP_PX`), tiles placed with no gap: the final mosaic.
A doubled or broken edge at a seam is residual misalignment. To see the overlap
itself, use sections 13/14.

1. **9x9 FOVs** around section 11's centre FOV.
2. **3x3 FOVs** (centre + 8 neighbours) at native resolution.
3. **The 4 corners of the centre FOV** -- a `MOSAIC_2X2_ZOOM_HALF_UM`-wide
   crop where it meets two side neighbours and one diagonal (NW/NE/SW/SE).

In [ ]:
center_xy      = full_positions[anchor_fov]
half_window_um = MOSAIC_9X9_N_FOVS * STEP_SIZE_UM / 2
window_fov_ids = [
    f for f, (x, y) in full_positions.items()
    if abs(x - center_xy[0]) <= half_window_um and abs(y - center_xy[1]) <= half_window_um
]
print(f"9x9 window: {len(window_fov_ids)} FOV(s) around FOV {anchor_fov} "
      f"({center_xy[0]:.1f}, {center_xy[1]:.1f}) um "
      f"(+/-{half_window_um:.1f} um -- MOSAIC_9X9_N_FOVS={MOSAIC_9X9_N_FOVS}).")

window_raw_frames, _ = load_raw_frames_for_round(
    target_round_id, meta, reference_frame_idx, fov_subset=window_fov_ids,
    frame_width=config.frame_width, frame_height=config.frame_height,
)
# load_raw_frames_for_round reads raw frames; apply ORIENT_* here too.
window_raw_frames = {
    fov_id: apply_microscope_orientation(frame, transpose=ORIENT_TRANSPOSE,
                                          flip_horizontal=ORIENT_FLIP_HORIZONTAL, flip_vertical=ORIENT_FLIP_VERTICAL)
    for fov_id, frame in window_raw_frames.items()
}
print(f"Loaded {len(window_raw_frames)} raw DAPI frame(s) for the 9x9 window.")


In [ ]:
# Overlap-border crop from the measured grid (section 3), not
# ExperimentConfig's formula. Overrides the placeholder MOSAIC_CROP_PX=0.
MOSAIC_CROP_PX = int(round(FRAME_WIDTH_PX * OVERLAP_FRACTION / 2))
print(f"MOSAIC_CROP_PX = {MOSAIC_CROP_PX} px (overlap fraction {OVERLAP_FRACTION:.3f}, "
      f"frame width {FRAME_WIDTH_PX} px)")

window_original_positions  = {f: full_positions[f] for f in window_raw_frames}
window_corrected_positions = {f: corrected_positions[f] for f in window_raw_frames}

window_original_canvas = create_mosaic_ffc(
    window_raw_frames, window_original_positions,
    CACHE_DIR / f"mosaic_round{target_round_id}_9x9_original.png",
    ffc_field=ffc_field, crop_px=MOSAIC_CROP_PX, padding=0,
    thumbnail_size=MOSAIC_THUMBNAIL_SIZE, flip_y=FLIP_Y,
)
window_corrected_canvas = create_mosaic_ffc(
    window_raw_frames, window_corrected_positions,
    CACHE_DIR / f"mosaic_round{target_round_id}_9x9_corrected.png",
    ffc_field=ffc_field, crop_px=MOSAIC_CROP_PX, padding=0,
    thumbnail_size=MOSAIC_THUMBNAIL_SIZE, flip_y=FLIP_Y,
)

# 3x3 block at native resolution: passing the real frame size skips resizing.
block3x3_raw_frames_by_fov = {fov_id: block_3x3_frames[name] for name, fov_id in block_3x3.items()}
block3x3_original_by_fov   = {fov_id: full_positions[fov_id] for fov_id in block_3x3.values()}
block3x3_corrected_by_fov  = {fov_id: corrected_positions[fov_id] for fov_id in block_3x3.values()}

block3x3_original_canvas = create_mosaic_ffc(
    block3x3_raw_frames_by_fov, block3x3_original_by_fov,
    CACHE_DIR / f"mosaic_round{target_round_id}_3x3_original.png",
    ffc_field=ffc_field, crop_px=MOSAIC_CROP_PX, padding=0,
    thumbnail_size=(FRAME_WIDTH_PX, FRAME_WIDTH_PX), flip_y=FLIP_Y,
)
block3x3_corrected_canvas = create_mosaic_ffc(
    block3x3_raw_frames_by_fov, block3x3_corrected_by_fov,
    CACHE_DIR / f"mosaic_round{target_round_id}_3x3_corrected.png",
    ffc_field=ffc_field, crop_px=MOSAIC_CROP_PX, padding=0,
    thumbnail_size=(FRAME_WIDTH_PX, FRAME_WIDTH_PX), flip_y=FLIP_Y,
)
print(f"3x3 mosaic canvases: original={block3x3_original_canvas.shape}, "
      f"corrected={block3x3_corrected_canvas.shape}")

In [ ]:
def crop_at(canvas, center_row_px, center_col_px, half_um, pixel_size_um):
    h, w = canvas.shape[:2]
    half_px = int(round(half_um / pixel_size_um))
    r0, r1 = max(center_row_px - half_px, 0), min(center_row_px + half_px, h)
    c0, c1 = max(center_col_px - half_px, 0), min(center_col_px + half_px, w)
    return canvas[r0:r1, c0:c1]


def four_corner_crops(canvas, half_um, pixel_size_um):
    """The 3x3 canvas is at NATIVE resolution with all nine tiles the same
    size, laid out in 3 equal rows/columns -- the four points where the
    centre FOV meets two 4-connected neighbours and one diagonal neighbour
    (its own NW/NE/SW/SE corners) sit exactly at 1/3 and 2/3 of the canvas's
    height/width."""
    h, w = canvas.shape[:2]
    row_third, col_third = h / 3, w / 3
    corners = {
        "NW": (row_third, col_third), "NE": (row_third, 2 * col_third),
        "SW": (2 * row_third, col_third), "SE": (2 * row_third, 2 * col_third),
    }
    return {
        name: crop_at(canvas, int(round(r)), int(round(c)), half_um, pixel_size_um)
        for name, (r, c) in corners.items()
    }


block3x3_original_corners  = four_corner_crops(block3x3_original_canvas, MOSAIC_2X2_ZOOM_HALF_UM, PIXEL_SIZE_UM)
block3x3_corrected_corners = four_corner_crops(block3x3_corrected_canvas, MOSAIC_2X2_ZOOM_HALF_UM, PIXEL_SIZE_UM)
print(f"Corner crop shapes: {[(k, v.shape) for k, v in block3x3_original_corners.items()]}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, canvas, title in ((axes[0], window_original_canvas, "9x9 -- original"),
                           (axes[1], window_corrected_canvas, "9x9 -- corrected")):
    ax.imshow(canvas, cmap="gray")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
fig.suptitle(f"9x9 FOV window around FOV {anchor_fov} -- round {target_round_id}",
             fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.mosaic_9x9.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, canvas, title in ((axes[0], block3x3_original_canvas, "3x3 -- original"),
                           (axes[1], block3x3_corrected_canvas, "3x3 -- corrected")):
    ax.imshow(canvas, cmap="gray")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
fig.suptitle(f"3x3 FOV block around FOV {anchor_fov} (native resolution) -- round {target_round_id}",
             fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.mosaic_3x3.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(9, 18))
for row, corner in enumerate(("NW", "NE", "SW", "SE")):
    for col, (corners_dict, title) in enumerate(
        ((block3x3_original_corners, "original"), (block3x3_corrected_corners, "corrected"))
    ):
        ax = axes[row][col]
        ax.imshow(corners_dict[corner], cmap="gray")
        ax.set_title(f"{corner} -- {title}", fontsize=PLOT_TICK_FONTSIZE)
        ax.axis("off")

fig.suptitle(f"The 4 corners of FOV {anchor_fov} (+/-{MOSAIC_2X2_ZOOM_HALF_UM:.0f} um) -- "
             f"before vs. after correction", fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.mosaic_4corners.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 13 — Plot: colour overlap overlay of the 2x2 corner block

Same corner block and FFC field as section 12, but each FOV is placed at its
real position, tinted one colour (red/green/blue/yellow) and added, so
overlaps are drawn on top of each other. Aligned content blends into one hue;
misaligned content shows colour fringing.

In [ ]:
def build_corner_overlay(frames, positions_um, pixel_size_um, colors, flip_y, ffc_field=None):
    """Place FFC-corrected frames at their real stage positions (nominal or
    corrected), each tinted a distinct colour, combined ADDITIVELY (then
    clipped) so a genuinely overlapping region shows real colour mixing --
    unlike a max-combine, where the brighter tile simply hides the other."""
    from MERci.analysis.ffc import apply_ffc

    h, w = next(iter(frames.values())).shape
    ys_sign = -1.0 if flip_y else 1.0
    xs = [positions_um[n][0] for n in frames]
    ys = [ys_sign * positions_um[n][1] for n in frames]
    x_min = min(xs) - w * pixel_size_um / 2
    y_min = min(ys) - h * pixel_size_um / 2
    x_max = max(xs) + w * pixel_size_um / 2
    y_max = max(ys) + h * pixel_size_um / 2
    canvas_w = int(round((x_max - x_min) / pixel_size_um))
    canvas_h = int(round((y_max - y_min) / pixel_size_um))
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.float32)
    for name, img in frames.items():
        corrected_img = apply_ffc(img, ffc_field) if ffc_field is not None else img.astype(np.float32)
        lo, hi = np.percentile(corrected_img, [1, 99.5])
        norm = np.clip((corrected_img - lo) / max(hi - lo, 1e-6), 0, 1)
        x_um, y_um = positions_um[name][0], ys_sign * positions_um[name][1]
        col0 = int(round((x_um - w * pixel_size_um / 2 - x_min) / pixel_size_um))
        row0 = int(round((y_um - h * pixel_size_um / 2 - y_min) / pixel_size_um))
        tile_rgb = norm[..., None] * np.array(colors[name], dtype=np.float32)
        canvas[row0:row0 + h, col0:col0 + w] += tile_rgb
    return np.clip(canvas, 0, 1)


BLOCK_COLORS = {"anchor": (1.0, 0.0, 0.0), "right": (0.0, 1.0, 0.0),
                "up": (0.0, 0.0, 1.0), "diag": (1.0, 1.0, 0.0)}

overlay_original_positions  = {name: full_positions[fov_id] for name, fov_id in block_names.items()}
overlay_corrected_positions = {name: corrected_positions[fov_id] for name, fov_id in block_names.items()}

overlay_before = build_corner_overlay(block_frames, overlay_original_positions, PIXEL_SIZE_UM,
                                       BLOCK_COLORS, FLIP_Y, ffc_field=ffc_field)
overlay_after  = build_corner_overlay(block_frames, overlay_corrected_positions, PIXEL_SIZE_UM,
                                       BLOCK_COLORS, FLIP_Y, ffc_field=ffc_field)

print(f"Overlay canvases built: before={overlay_before.shape[:2]}, after={overlay_after.shape[:2]}")


In [ ]:
overlay_before_zoom = crop_at(overlay_before, overlay_before.shape[0] // 2, overlay_before.shape[1] // 2,
                               CORNER_ZOOM_HALF_UM, PIXEL_SIZE_UM)
overlay_after_zoom  = crop_at(overlay_after, overlay_after.shape[0] // 2, overlay_after.shape[1] // 2,
                               CORNER_ZOOM_HALF_UM, PIXEL_SIZE_UM)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, crop, title in ((axes[0], overlay_before_zoom, "before correction"),
                         (axes[1], overlay_after_zoom, "after correction")):
    ax.imshow(crop)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
handles = [mpatches.Patch(color=c, label=n) for n, c in BLOCK_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=PLOT_LEGEND_FONTSIZE)
fig.suptitle(f"Colour overlap overlay of FOVs {block_fov_ids} (8-connected) -- "
             f"before vs. after correction", fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig_path = figures_dir / f"{NOTEBOOK_NAME}.corner_overlay.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 14 -- Plot: 4-direction overlap overlay (centre FOV vs. each cardinal neighbour, before vs. after correction)

Section 13's overlay with 2 tiles (anchor + one neighbour) per cardinal
direction, at the nominal and corrected positions.

The centre FOV is the training anchor with kept correspondences in all 4
directions and the highest minimum correlation peak across them, so every
side has real signal.

In [ ]:
def _overlap_correlation_peak(a_crop, n_crop):
    """FFT cross-power correlation peak (0-1, higher = stronger real match) --
    used only to PICK a well-signalled example FOV for this diagnostic below, not
    as part of the actual registration/fit above (that already happened in
    section 5/6 via phase_drift)."""
    f0 = np.fft.fft2(remove_hot_pixels(a_crop).astype(np.float64))
    f1 = np.fft.fft2(remove_hot_pixels(n_crop).astype(np.float64))
    cross = f0 * np.conj(f1)
    cross /= np.abs(cross) + 1e-10
    return float(np.abs(np.fft.ifft2(cross)).max())


def oriented_frame(fov_id):
    """load_dapi_frame (section 3) is a raw loader -- shared with the fit/audit
    path above, which applies orientation internally -- so this diagnostic must
    apply it explicitly itself, same as block_3x3_frames in section 10."""
    return apply_microscope_orientation(load_dapi_frame(fov_id), transpose=ORIENT_TRANSPOSE,
                                         flip_horizontal=ORIENT_FLIP_HORIZONTAL, flip_vertical=ORIENT_FLIP_VERTICAL)


# Candidate centres: training anchors with a KEPT (non-outlier) correspondence in
# all 4 cardinal directions -- pick whichever has real signal on every side.
by_anchor = {}
for c in correspondences:
    by_anchor.setdefault(c.anchor_fov, {})[c.direction] = c.neighbor_fov
complete_anchors = {a: nbs for a, nbs in by_anchor.items()
                    if set(nbs) == {"up", "down", "left", "right"}}
if not complete_anchors:
    raise RuntimeError("No training anchor has all 4 cardinal directions kept -- "
                       "try a larger N_ANCHORS or MAD_THRESHOLD.")

FOUR_DIRECTION_CENTER, best_min_peak, FOUR_DIRECTION_NEIGHBORS = None, -1.0, None
for center_candidate, nbs in complete_anchors.items():
    center_img = oriented_frame(center_candidate)
    peaks = {
        direction: _overlap_correlation_peak(
            *crop_overlap(center_img, oriented_frame(nb_fov), direction, OVERLAP_FRACTION))
        for direction, nb_fov in nbs.items()
    }
    min_peak = min(peaks.values())
    if min_peak > best_min_peak:
        FOUR_DIRECTION_CENTER, best_min_peak, FOUR_DIRECTION_NEIGHBORS = center_candidate, min_peak, nbs

print(f"4-direction overlay centre FOV: {FOUR_DIRECTION_CENTER} (min cross-power peak "
      f"{best_min_peak:.4f} across all 4 neighbours: {FOUR_DIRECTION_NEIGHBORS})")


In [ ]:
FOUR_DIRECTION_COLORS = {"anchor": (1.0, 0.0, 0.0), "neighbor": (0.0, 1.0, 0.0)}

fig, axes = plt.subplots(4, 2, figsize=(9, 18))
for row, direction in enumerate(("up", "down", "left", "right")):
    nb_fov = FOUR_DIRECTION_NEIGHBORS[direction]
    pair_frames = {"anchor": oriented_frame(FOUR_DIRECTION_CENTER), "neighbor": oriented_frame(nb_fov)}

    nominal_pair   = {"anchor": full_positions[FOUR_DIRECTION_CENTER], "neighbor": full_positions[nb_fov]}
    corrected_pair = {"anchor": corrected_positions[FOUR_DIRECTION_CENTER], "neighbor": corrected_positions[nb_fov]}

    overlay_pair_nominal   = build_corner_overlay(pair_frames, nominal_pair, PIXEL_SIZE_UM,
                                                   FOUR_DIRECTION_COLORS, FLIP_Y, ffc_field=ffc_field)
    overlay_pair_corrected = build_corner_overlay(pair_frames, corrected_pair, PIXEL_SIZE_UM,
                                                   FOUR_DIRECTION_COLORS, FLIP_Y, ffc_field=ffc_field)

    # Zoom to the shared edge (the canvas centre for 2 tiles).
    zoom_nominal = crop_at(overlay_pair_nominal, overlay_pair_nominal.shape[0] // 2,
                           overlay_pair_nominal.shape[1] // 2, CORNER_ZOOM_HALF_UM, PIXEL_SIZE_UM)
    zoom_corrected = crop_at(overlay_pair_corrected, overlay_pair_corrected.shape[0] // 2,
                             overlay_pair_corrected.shape[1] // 2, CORNER_ZOOM_HALF_UM, PIXEL_SIZE_UM)

    axes[row, 0].imshow(zoom_nominal)
    axes[row, 0].set_title(f"{direction}: anchor {FOUR_DIRECTION_CENTER} vs neighbor {nb_fov}\n"
                           f"nominal (before correction)", fontsize=PLOT_TITLE_FONTSIZE - 2)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(zoom_corrected)
    axes[row, 1].set_title(f"{direction}: after correction", fontsize=PLOT_TITLE_FONTSIZE - 2)
    axes[row, 1].axis("off")

handles = [mpatches.Patch(color=color, label=name) for name, color in FOUR_DIRECTION_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=PLOT_LEGEND_FONTSIZE)
fig.suptitle(f"4-direction overlap overlay around FOV {FOUR_DIRECTION_CENTER} -- before vs. after correction\n"
             "(red=anchor, green=neighbour -- yellow/overlap = well aligned)", fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout(rect=[0, 0.04, 1, 0.95])
fig_path = figures_dir / f"{NOTEBOOK_NAME}.four_direction_overlay.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 15 — Validate the correction on held-out FOVs

Tests whether one global affine explains almost all of the misalignment,
leaving only small random jitter. A fresh set of `N_VALIDATION_ANCHORS`
anchors, disjoint from the fit's, is registered as in section 5. Each
neighbour's measured position is compared with what the fitted correction
predicts from its nominal position.

If the model holds, `error_after` is much smaller than `error_before`, with no
spatial pattern in the residuals.

In [ ]:
VALIDATION_CACHE = CACHE_DIR / "validation_correspondences.csv"

# Disjoint from the training anchors (not from their neighbours).
train_anchor_fovs    = set(correspondences_df["anchor_fov"].unique())
validation_candidates = [f for f in fov_order if f not in train_anchor_fovs]

if VALIDATION_CACHE.exists() and not FORCE_RECOMPUTE:
    validation_df = pd.read_csv(VALIDATION_CACHE)
    print(f"Loaded cached validation correspondences: {VALIDATION_CACHE}  ({len(validation_df)} rows)")
else:
    n_validation_actual = min(N_VALIDATION_ANCHORS, len(validation_candidates))
    reporter = ProgressReporter(total=n_validation_actual * 4, label="Registering held-out neighbour pairs")

    def _progress_validation(done, total):
        reporter.update(done - reporter.n_done)

    validation_correspondences = sample_neighbor_correspondences(
        fov_ids=validation_candidates,
        positions=full_positions,
        load_frame=load_dapi_frame,
        step_size_um=STEP_SIZE_UM,
        pixel_size_um=PIXEL_SIZE_UM,
        overlap_fraction=OVERLAP_FRACTION,
        n_anchors=N_VALIDATION_ANCHORS,
        tolerance_fraction=TOLERANCE_FRACTION,
        upsample_factor=UPSAMPLE_FACTOR,
        orient_transpose=ORIENT_TRANSPOSE,
        orient_flip_horizontal=ORIENT_FLIP_HORIZONTAL,
        orient_flip_vertical=ORIENT_FLIP_VERTICAL,
        seed=VALIDATION_SEED,
        progress_callback=_progress_validation,
    )
    reporter.done()

    validation_df = pd.DataFrame([{
        "anchor_fov": c.anchor_fov, "neighbor_fov": c.neighbor_fov, "direction": c.direction,
        "nominal_x": c.nominal_xy[0], "nominal_y": c.nominal_xy[1],
        "measured_x": c.measured_xy[0], "measured_y": c.measured_xy[1],
        "error": c.error,
    } for c in validation_correspondences])
    validation_df.to_csv(VALIDATION_CACHE, index=False)
    print(f"Sampled {len(validation_df)} held-out correspondence(s) from "
          f"{n_validation_actual} validation anchor FOV(s), disjoint from the "
          f"{len(train_anchor_fovs)} training anchor(s). Saved: {VALIDATION_CACHE}")

print(validation_df.head())


In [ ]:
validation_correspondences_all = [
    NeighborCorrespondence(
        anchor_fov=int(r.anchor_fov), neighbor_fov=int(r.neighbor_fov), direction=r.direction,
        nominal_xy=(r.nominal_x, r.nominal_y), measured_xy=(r.measured_x, r.measured_y), error=r.error,
    )
    for r in validation_df.itertuples()
]
validation_kept, validation_rejected = filter_correspondence_outliers(
    validation_correspondences_all, mad_threshold=MAD_THRESHOLD,
)
print(f"Kept {len(validation_kept)}/{len(validation_correspondences_all)} held-out correspondence(s) "
      f"({len(validation_rejected)} rejected as outliers -- same rule as section 6, applied "
      f"independently here since a bad registration is a bad MEASUREMENT of ground truth, "
      f"not a failure of the correction being tested).")
for c in validation_rejected:
    shift_um = float(np.hypot(c.measured_xy[0] - c.nominal_xy[0], c.measured_xy[1] - c.nominal_xy[1]))
    print(f"  rejected: anchor={c.anchor_fov} neighbor={c.neighbor_fov} "
          f"direction={c.direction!r} shift={shift_um:.2f} um")

validation_df = pd.DataFrame([{
    "anchor_fov": c.anchor_fov, "neighbor_fov": c.neighbor_fov, "direction": c.direction,
    "nominal_x": c.nominal_xy[0], "nominal_y": c.nominal_xy[1],
    "measured_x": c.measured_xy[0], "measured_y": c.measured_xy[1],
    "error": c.error,
} for c in validation_kept])

val_nominal   = validation_df[["nominal_x", "nominal_y"]].to_numpy()
val_measured  = validation_df[["measured_x", "measured_y"]].to_numpy()
val_corrected = correction.transform_points(val_nominal)

validation_df["error_before_um"] = np.linalg.norm(val_nominal - val_measured, axis=1)
validation_df["error_after_um"]  = np.linalg.norm(val_corrected - val_measured, axis=1)
validation_df["residual_dx_um"]  = val_corrected[:, 0] - val_measured[:, 0]
validation_df["residual_dy_um"]  = val_corrected[:, 1] - val_measured[:, 1]

median_before = float(validation_df["error_before_um"].median())
median_after  = float(validation_df["error_after_um"].median())

print(f"\nHeld-out position error (n={len(validation_df)} correspondences, "
      f"{validation_df['anchor_fov'].nunique()} FOV(s) not used in the fit):")
print(f"  before correction : median {median_before:.3f} um, "
      f"mean {validation_df['error_before_um'].mean():.3f} um, "
      f"max {validation_df['error_before_um'].max():.3f} um")
print(f"  after correction  : median {median_after:.3f} um, "
      f"mean {validation_df['error_after_um'].mean():.3f} um, "
      f"max {validation_df['error_after_um'].max():.3f} um")
if median_before > 0:
    print(f"  median reduction   : {100 * (1 - median_after / median_before):.1f}%")

if median_after >= median_before:
    print(
        f"\nNOTE: the correction did not reduce held-out position error. This does NOT "
        "necessarily mean the method failed -- it means that, for THIS round/microscope, "
        "the systematic camera-vs-stage rotation is smaller than the random, per-FOV "
        "positioning/registration noise floor (here, a few um). Fitting an affine to noise "
        "cannot improve on it; a real, larger rotation would show a clear reduction instead. "
        f"Check camera_rotation_correction_{POSITIONS_TAG}.json's fitted rotation/scale -- a "
        "value very close to identity (0 deg, scale 1.0) supports this reading."
    )


In [ ]:
error_max = max(validation_df["error_before_um"].max(), validation_df["error_after_um"].max())
bins = np.linspace(0, error_max * 1.05, 20)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(validation_df["error_before_um"], bins=bins, alpha=0.6, color="tab:blue",
        label=f"before correction (median {median_before:.2f} um)")
ax.hist(validation_df["error_after_um"], bins=bins, alpha=0.6, color="tab:red",
        label=f"after correction (median {median_after:.2f} um)")
ax.set_xlabel("position error vs. independently-measured true position (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("count", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Held-out validation -- {len(validation_df)} correspondence(s), "
             f"{validation_df['anchor_fov'].nunique()} FOV(s) not in fit",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()

fig_path = figures_dir / f"{NOTEBOOK_NAME}.validation_error_histogram.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


In [ ]:
residual_mag = np.hypot(validation_df["residual_dx_um"], validation_df["residual_dy_um"])
plot_extent  = max(validation_df["measured_x"].max() - validation_df["measured_x"].min(),
                    validation_df["measured_y"].max() - validation_df["measured_y"].min())
arrow_scale  = (0.05 * plot_extent) / max(float(residual_mag.max()), 1e-9)

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(validation_df["measured_x"], validation_df["measured_y"],
                 c=residual_mag, cmap="viridis", s=70, zorder=3)
ax.quiver(validation_df["measured_x"], validation_df["measured_y"],
          validation_df["residual_dx_um"] * arrow_scale, validation_df["residual_dy_um"] * arrow_scale,
          angles="xy", scale_units="xy", scale=1, color="k", width=0.004, alpha=0.7, zorder=2)
ax.invert_yaxis()
ax.axis("equal")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("residual error after correction (um)", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Held-out residual error after correction (arrows exaggerated {arrow_scale:.0f}x)\n"
             f"a spatial pattern here would mean the affine model is wrong",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()

fig_path = figures_dir / f"{NOTEBOOK_NAME}.validation_residual_map.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")
